# Import Segment

In [ ]:
import numpy
import pandas
import tensorflow
import sklearn
import matplotlib
import seaborn
import os
import cv2

import sklearn.model_selection

# Data Preprocessing

In [ ]:
NIH_Data_Entry = pandas.read_csv("/kaggle/input/data/Data_Entry_2017.csv")

In [ ]:
NIH_Data_Entry.head()

In [ ]:
NIH_Data_Entry.info()

In [ ]:
# empty column
NIH_Data_Entry.drop("Unnamed: 11", axis=1, inplace=True)

In [ ]:
NIH_Data_Entry.info()

In [ ]:
# select only related columns.
NIH_Data_Entry = NIH_Data_Entry[["Image Index", "Finding Labels"]]

In [ ]:
NIH_Data_Entry.info()

In [ ]:
# get all possible values of finding labels
entries: set = set()
for index, row in NIH_Data_Entry.iterrows():
    entries.update(row["Finding Labels"].split("|"))
entries

In [ ]:
# exclude 'No Finding' as it means non of the 14 classes
unknown_class: str = "No Finding"
entries.remove(unknown_class)

In [ ]:
# construct possible classes from the last set
classes_names: list[str] = list()
for entry in entries:
    if entry not in classes_names:
        classes_names.append(entry)
classes_names = sorted(classes_names)
classes_names

In [ ]:
def is_label_exist(cell: str, label: str) -> bool:
    return label in cell.split("|")

In [ ]:
# process dataset "Finding Labels" column
for class_name in classes_names:
    NIH_Data_Entry[class_name] = NIH_Data_Entry["Finding Labels"].apply(lambda cell: is_label_exist(cell, class_name))

In [ ]:
NIH_Data_Entry.head()

In [ ]:
NIH_Data_Entry.tail()

In [ ]:
# drop "Finding Labels" column
NIH_Data_Entry.drop("Finding Labels", axis=1, inplace=True)

# Split Dataset

In [ ]:
with open("/kaggle/input/data/train_val_list.txt") as train_val_txt:
  train_val_indices = numpy.array(train_val_txt.read().split("\n"))
train_val_indices

In [ ]:
with open("/kaggle/input/data/test_list.txt") as test_txt:
  test_indices = numpy.array(test_txt.read().split("\n"))
test_indices

In [ ]:
features_train = NIH_Data_Entry[NIH_Data_Entry["Image Index"].isin(train_val_indices)]["Image Index"].values
targets_train = NIH_Data_Entry[NIH_Data_Entry["Image Index"].isin(train_val_indices)].drop("Image Index", axis=1).values
features_test = NIH_Data_Entry[NIH_Data_Entry["Image Index"].isin(test_indices)]["Image Index"].values
targets_test = NIH_Data_Entry[NIH_Data_Entry["Image Index"].isin(test_indices)].drop("Image Index", axis=1).values

In [ ]:
# split to training and validation sets.
features_train, features_validation, targets_train, targets_validation = sklearn.model_selection.train_test_split(features_train, targets_train, test_size=0.2)

In [ ]:
print("features_train shape:", features_train.shape)
print("targets_train shape:", targets_train.shape)
print("features_validation shape:", features_validation.shape)
print("targets_validation shape:", targets_validation.shape)
print("features_test shape:", features_test.shape)
print("targets_test shape:", targets_test.shape)

# Input Processing

In [ ]:
class NIHDataGenerator(tensorflow.keras.utils.Sequence):
    def __init__(self, image_indices: numpy.ndarray, targets: numpy.ndarray, batch_size: int, search_dirs: list[str], image_size: tuple[int, int] = (256, 256), channels: int = 3, image_size_reduction: int = 0, max_image_rotation: float = 0, noise_std: float = 0.02, **kwargs) -> None:
        super().__init__(**kwargs)
        self.image_indices: numpy.ndarray = image_indices
        self.targets: numpy.ndarray = targets
        self.batch_size: int = batch_size

        self.image_size: tuple[int, int] = image_size
        self.channels: int = channels
        self.image_size_reduction: int = image_size_reduction
        self.search_dirs: list[str] = search_dirs
        self.max_image_rotation: float = abs(max_image_rotation)
        self.noise_std: float = noise_std

        self.derived_size: tuple[int, int] = self.get_derived_image_shape()

    def __len__(self) -> int:
        return int(numpy.ceil(len(self.image_indices) / float(self.batch_size)))

    def __getitem__(self, index) -> tuple[numpy.ndarray, numpy.ndarray]:
        # calculate indices
        lower_index: int = index * self.batch_size
        upper_index: int = min(lower_index + self.batch_size, len(self.image_indices))
        batch_indices: numpy.ndarray = self.image_indices[lower_index:upper_index]
        
        # read images files
        batch_images: list[numpy.ndarray] = list()
        for image_index in batch_indices:
            for search_dir in self.search_dirs:
                image_path: str = os.path.join(search_dir, image_index)
                if os.path.exists(image_path):
                    image: numpy.ndarray = cv2.imread(image_path)
                    if image is not None:
                        batch_images.append(self._process_image(image))
                    break
        batch_images: numpy.ndarray = numpy.array(batch_images)
        
        # slice targets
        batch_targets: numpy.ndarray = self.targets[lower_index:upper_index]
        return batch_images, batch_targets

    def _process_image(self, image: numpy.ndarray) -> numpy.ndarray:
        # convert to grayscale
        if self.channels == 1:
          image: numpy.ndarray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        # resize image
        image: numpy.ndarray = cv2.resize(image, self.derived_size)
        # normalize image
        image: numpy.ndarray = image / 255
        # ratate image by random degree using self.max_image_rotation
        if self.max_image_rotation > 0:
            rotation_degree: float = numpy.random.uniform(-self.max_image_rotation, self.max_image_rotation)
            rotation_matrix: numpy.ndarray = cv2.getRotationMatrix2D((self.derived_size[0] / 2, self.derived_size[1] / 2), rotation_degree, 1) # center of rotation, degree, scaling factor
            image: numpy.ndarray = cv2.warpAffine(image, rotation_matrix, self.derived_size)
        # apply noise
        image: numpy.ndarray = image + numpy.random.normal(0, self.noise_std, image.shape)
        
        return image
    def get_derived_image_shape(self) -> tuple[int, int]:
        return (self.image_size[0] // (2**self.image_size_reduction), self.image_size[1] // (2**self.image_size_reduction))


# Model Specification

In [ ]:
search_dirs: list = ["/kaggle/input/data/images_001/images", "/kaggle/input/data/images_002/images", "/kaggle/input/data/images_003/images",
                     "/kaggle/input/data/images_004/images", "/kaggle/input/data/images_005/images", "/kaggle/input/data/images_006/images",
                     "/kaggle/input/data/images_007/images", "/kaggle/input/data/images_008/images", "/kaggle/input/data/images_009/images",
                     "/kaggle/input/data/images_010/images", "/kaggle/input/data/images_011/images", "/kaggle/input/data/images_012/images",
                    ]

In [ ]:
load_size: int = 80000
NIH_train_generator: NIHDataGenerator = NIHDataGenerator(features_train, targets_train, load_size, search_dirs, channels=1, image_size_reduction = 2, max_image_rotation=10, noise_std=0.01)
NIH_validation_generator: NIHDataGenerator = NIHDataGenerator(features_validation, targets_validation, int(load_size/4), search_dirs, channels=1, image_size_reduction = 2, max_image_rotation=5, noise_std=0.02)
NIH_test_generator: NIHDataGenerator = NIHDataGenerator(features_test, targets_test, load_size, search_dirs, channels=1, image_size_reduction = 2, max_image_rotation=5, noise_std=0.03)

In [ ]:
# load dataset to memory
features_train, targets_train = NIH_train_generator.__getitem__(0)
features_validation, targets_validation = NIH_validation_generator.__getitem__(0)
features_test, targets_test = NIH_test_generator.__getitem__(0)

In [ ]:
learning_rate: float = 0.05

In [ ]:
NIH_model: tensorflow.keras.Sequential = tensorflow.keras.Sequential()

# input layer
NIH_model.add(tensorflow.keras.layers.Input(shape=(NIH_train_generator.derived_size[0], NIH_train_generator.derived_size[1], 1)))

# first convolution layer
NIH_model.add(tensorflow.keras.layers.Conv2D(128, (2, 2), activation='relu'))
NIH_model.add(tensorflow.keras.layers.MaxPooling2D((2, 2)))

# second convolution layer
NIH_model.add(tensorflow.keras.layers.Conv2D(256, (4, 4), activation='relu'))
NIH_model.add(tensorflow.keras.layers.MaxPooling2D((2, 2)))

NIH_model.add(tensorflow.keras.layers.Conv2D(64, (2, 2), activation='relu'))

NIH_model.add(tensorflow.keras.layers.Flatten())
NIH_model.add(tensorflow.keras.layers.Dropout(0.25))

NIH_model.add(tensorflow.keras.layers.Dense(512, activation='relu'))
NIH_model.add(tensorflow.keras.layers.Dropout(0.25))

NIH_model.add(tensorflow.keras.layers.Dense(128, activation='relu'))
NIH_model.add(tensorflow.keras.layers.Dropout(0.25))

NIH_model.add(tensorflow.keras.layers.Dense(14, activation='sigmoid'))

NIH_model.compile(optimizer=tensorflow.keras.optimizers.Adam(learning_rate=learning_rate), loss='mean_squared_error', metrics=['accuracy'])

In [ ]:
# adding callbacks to the training process
callbacks = [
    tensorflow.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
    tensorflow.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=0.00001),
    tensorflow.keras.callbacks.ModelCheckpoint(filepath='model.keras', monitor='val_loss', save_best_only=True)
]

In [ ]:
targets_train = targets_train.astype(float)
targets_validation = targets_validation.astype(float)

In [ ]:
NIH_model.fit(features_train, targets_train, validation_data=(features_validation, targets_validation), epochs=1000, batch_size=1024, callbacks=callbacks)

In [ ]:
def get_accuracy(predictions, targets, threshold = 0.8):
    actual_predictions = targets.astype(bool)
    correct_predictions = ((predictions >= threshold) == actual_predictions).min(axis = 1).mean()
    return correct_predictions

In [ ]:
# get the best threshold
def get_best_threshold(predictions: numpy.ndarray, targets: numpy.ndarray, get_accuracy: callable):
  best_threshold: float = 0
  best_accuracy: float = 0
  for threshold in numpy.arange(0, 1, 0.01):
      accuracy: float = get_accuracy(predictions, targets, threshold)
      if accuracy > best_accuracy:
          best_accuracy = accuracy
          best_threshold = threshold
  return best_threshold, best_accuracy

In [ ]:
NIH_predictions = NIH_model.predict(features_validation)

In [ ]:
threshold, accuracy = get_best_threshold(NIH_predictions, targets_validation, get_accuracy)
print(threshold)
print(accuracy)

In [ ]:
NIH_test_predictions = NIH_model.predict(features_test)
print(get_accuracy(NIH_test_predictions, targets_test, threshold))